###0)

In [0]:
from datetime import (
    datetime,
    date,
    time,
    timezone,
)
from zoneinfo import ZoneInfo

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)

dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

EVALUATION_START_DATE_TEXT = dbutils.widgets.get(
    "evaluation_start_date_local"
)

EVALUATION_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

evaluation_start_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_START_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

evaluation_end_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_END_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

if evaluation_end_local <= evaluation_start_local:
    raise ValueError(
        "Data końcowa musi być późniejsza "
        "od daty początkowej."
    )

EVALUATION_START_UTC = (
    evaluation_start_local
    .astimezone(timezone.utc)
)

EVALUATION_END_UTC = (
    evaluation_end_local
    .astimezone(timezone.utc)
)

EXPECTED_DECISION_COUNT = int(
    (
        EVALUATION_END_UTC
        - EVALUATION_START_UTC
    ).total_seconds()
    / (15 * 60)
)

EXPECTED_FORECAST_POINTS_PER_DECISION = 24

EXPECTED_ASOF_ROW_COUNT = (
    EXPECTED_DECISION_COUNT
    * EXPECTED_FORECAST_POINTS_PER_DECISION
)

print(
    "Zakres Silver:",
    EVALUATION_START_UTC,
    "—",
    EVALUATION_END_UTC
)

print(
    "Oczekiwane momenty decyzyjne:",
    EXPECTED_DECISION_COUNT
)

print(
    "Oczekiwane rekordy as-of:",
    EXPECTED_ASOF_ROW_COUNT
)

###1) Standaryzacja prognoz

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

forecast_raw = spark.table(
    "workspace.energy_bronze."
    "open_meteo_forecast_runs_raw"
)

temperature_valid = (
    F.col("temperature_2m_c").isNotNull()
    & F.col("temperature_2m_c").between(-60.0, 60.0)
)

humidity_valid = (
    F.col("relative_humidity_2m_pct").isNotNull()
    & F.col("relative_humidity_2m_pct").between(
        0.0,
        100.0
    )
)

cloud_cover_valid = (
    F.col("cloud_cover_pct").isNotNull()
    & F.col("cloud_cover_pct").between(0.0, 100.0)
)

wind_speed_valid = (
    F.col("wind_speed_10m_ms").isNotNull()
    & F.col("wind_speed_10m_ms").between(0.0, 100.0)
)

wind_direction_valid = (
    F.col("wind_direction_10m_deg").isNotNull()
    & F.col("wind_direction_10m_deg").between(
        0.0,
        360.0
    )
)

shortwave_valid = (
    F.col("shortwave_radiation_instant_wm2").isNotNull()
    & F.col(
        "shortwave_radiation_instant_wm2"
    ).between(0.0, 1600.0)
)

gti_valid = (
    F.col(
        "global_tilted_irradiance_instant_wm2"
    ).isNotNull()
    & F.col(
        "global_tilted_irradiance_instant_wm2"
    ).between(0.0, 1600.0)
)

forecast_silver = forecast_raw.select(
    "source_system",
    "forecast_model",
    "forecast_run_id",
    "latitude",
    "longitude",
    "elevation_m",
    "run_time_utc",
    "available_at_utc",
    "forecast_valid_time_utc",
    "lead_time_from_run_hours",
    "effective_lead_time_hours",
    "is_usable_for_replay",

    F.col("temperature_2m_c").alias(
        "temperature_2m_c_raw"
    ),
    F.when(
        temperature_valid,
        F.col("temperature_2m_c")
    ).alias("temperature_2m_c"),

    F.col("relative_humidity_2m_pct").alias(
        "relative_humidity_2m_pct_raw"
    ),
    F.when(
        humidity_valid,
        F.col("relative_humidity_2m_pct")
    ).alias("relative_humidity_2m_pct"),

    F.col("cloud_cover_pct").alias(
        "cloud_cover_pct_raw"
    ),
    F.when(
        cloud_cover_valid,
        F.col("cloud_cover_pct")
    ).alias("cloud_cover_pct"),

    F.col("wind_speed_10m_ms").alias(
        "wind_speed_10m_ms_raw"
    ),
    F.when(
        wind_speed_valid,
        F.col("wind_speed_10m_ms")
    ).alias("wind_speed_10m_ms"),

    F.col("wind_direction_10m_deg").alias(
        "wind_direction_10m_deg_raw"
    ),
    F.when(
        wind_direction_valid,
        F.pmod(
            F.col("wind_direction_10m_deg"),
            F.lit(360.0)
        )
    ).alias("wind_direction_10m_deg"),

    F.col(
        "shortwave_radiation_instant_wm2"
    ).alias("shortwave_radiation_wm2_raw"),
    F.when(
        shortwave_valid,
        F.col("shortwave_radiation_instant_wm2")
    ).alias("shortwave_radiation_wm2"),

    F.col(
        "global_tilted_irradiance_instant_wm2"
    ).alias("forecast_gti_wm2_raw"),
    F.when(
        gti_valid,
        F.col(
            "global_tilted_irradiance_instant_wm2"
        )
    ).alias("forecast_gti_wm2"),

    "is_day",

    (
        ~temperature_valid
        | ~humidity_valid
        | ~cloud_cover_valid
        | ~wind_speed_valid
        | ~wind_direction_valid
        | ~shortwave_valid
        | ~gti_valid
    ).alias("has_missing_or_outlier"),

    (
        temperature_valid
        & cloud_cover_valid
        & wind_speed_valid
        & gti_valid
    ).alias("is_core_forecast_complete"),

    "_source_file_path",
    "_record_hash",
    "_ingested_at_utc"
)

display(forecast_silver.limit(10))

###2) Zapis Silver

In [0]:
(
    forecast_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.energy_silver."
        "open_meteo_forecast"
    )
)

print(
    "Zapisano: workspace.energy_silver."
    "open_meteo_forecast"
)

###3) Momenty decyzyjne dla zakresu

In [0]:
nlr_decisions = (
    spark.table(
        "workspace.energy_silver."
        "nlr_decision_features_15m"
    )
    .filter(
        (
            F.col("decision_time_utc")
            >= F.lit(
                EVALUATION_START_UTC
            ).cast("timestamp")
        )
        &
        (
            F.col("decision_time_utc")
            < F.lit(
                EVALUATION_END_UTC
            ).cast("timestamp")
        )
    )
    .select(
        "system_id",
        "decision_time_utc"
    )
    .dropDuplicates([
        "system_id",
        "decision_time_utc"
    ])
)

decision_profile_df = (
    nlr_decisions
    .agg(
        F.count("*").alias(
            "decision_count"
        ),
        F.countDistinct(
            "decision_time_utc"
        ).alias(
            "distinct_decision_time_count"
        ),
        F.min(
            "decision_time_utc"
        ).alias("min_decision_time_utc"),
        F.max(
            "decision_time_utc"
        ).alias("max_decision_time_utc")
    )
)

display(decision_profile_df)

decision_profile = (
    decision_profile_df
    .first()
    .asDict()
)

if (
    decision_profile["decision_count"]
    != EXPECTED_DECISION_COUNT
):
    raise RuntimeError(
        "Niepoprawna liczba momentów "
        "decyzyjnych: "
        f"{decision_profile['decision_count']} "
        "zamiast "
        f"{EXPECTED_DECISION_COUNT}."
    )

if (
    decision_profile[
        "distinct_decision_time_count"
    ]
    != EXPECTED_DECISION_COUNT
):
    raise RuntimeError(
        "Momenty decyzyjne nie są unikalne."
    )

print(
    "Miesięczna oś decyzji jest kompletna."
)

###4) Point-in-time join
##### available_at_utc <= decision_time_utc

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

forecast_silver = spark.table(
    "workspace.energy_silver."
    "open_meteo_forecast"
)

usable_forecasts = (
    forecast_silver
    .where(F.col("is_usable_for_replay"))
)

joined_candidates = (
    nlr_decisions.alias("decision")
    .join(
        F.broadcast(
            usable_forecasts.alias("forecast")
        ),
        (
            F.col("forecast.available_at_utc")
            <= F.col("decision.decision_time_utc")
        )
        & (
            F.col("forecast.forecast_valid_time_utc")
            > F.col("decision.decision_time_utc")
        )
        & (
            F.col("forecast.forecast_valid_time_utc")
            <= (
                F.col("decision.decision_time_utc")
                + F.expr("INTERVAL 24 HOURS")
            )
        ),
        "inner"
    )
    .select(
        F.col("decision.system_id"),
        F.col("decision.decision_time_utc"),

        F.col("forecast.forecast_model"),
        F.col("forecast.forecast_run_id"),
        F.col("forecast.run_time_utc"),
        F.col("forecast.available_at_utc"),
        F.col("forecast.forecast_valid_time_utc"),

        (
            F.unix_timestamp(
                F.col("forecast.forecast_valid_time_utc")
            )
            - F.unix_timestamp(
                F.col("decision.decision_time_utc")
            )
        ).cast("double").alias(
            "forecast_horizon_seconds"
        ),

        F.col("forecast.temperature_2m_c"),
        F.col("forecast.relative_humidity_2m_pct"),
        F.col("forecast.cloud_cover_pct"),
        F.col("forecast.wind_speed_10m_ms"),
        F.col("forecast.wind_direction_10m_deg"),
        F.col("forecast.shortwave_radiation_wm2"),
        F.col("forecast.forecast_gti_wm2"),
        F.col("forecast.is_day"),
        F.col("forecast.is_core_forecast_complete")
    )
)

latest_run_window = (
    Window
    .partitionBy(
        "system_id",
        "decision_time_utc",
        "forecast_valid_time_utc"
    )
    .orderBy(
        F.col("available_at_utc").desc(),
        F.col("run_time_utc").desc()
    )
)

weather_forecast_asof_hourly = (
    joined_candidates
    .withColumn(
        "_run_rank",
        F.row_number().over(latest_run_window)
    )
    .where(F.col("_run_rank") == 1)
    .drop("_run_rank")
    .withColumn(
        "forecast_horizon_minutes",
        F.col("forecast_horizon_seconds") / 60.0
    )
    .drop("forecast_horizon_seconds")
)

display(
    weather_forecast_asof_hourly
    .orderBy(
        "decision_time_utc",
        "forecast_valid_time_utc"
    )
    .limit(30)
)

###5) Zapis point-in-time Silver

In [0]:
(
    weather_forecast_asof_hourly.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.energy_silver."
        "open_meteo_forecast_asof_hourly"
    )
)

print(
    "Zapisano: workspace.energy_silver."
    "open_meteo_forecast_asof_hourly"
)

###6) Kontrola krzywych prognoz

In [0]:
from pyspark.sql import functions as F

forecast_asof = spark.table(
    "workspace.energy_silver."
    "open_meteo_forecast_asof_hourly"
)

curve_sizes = (
    forecast_asof
    .groupBy(
        "system_id",
        "decision_time_utc"
    )
    .agg(
        F.count("*").alias(
            "forecast_points"
        )
    )
)

curve_profile = (
    curve_sizes
    .agg(
        F.count("*").alias(
            "decision_count_with_forecast"
        ),
        F.min("forecast_points").alias(
            "min_points_per_decision"
        ),
        F.max("forecast_points").alias(
            "max_points_per_decision"
        ),
        F.avg("forecast_points").alias(
            "avg_points_per_decision"
        )
    )
)

display(curve_profile)

###7) Test data leakage

In [0]:
forecast_asof_df = spark.table(
    "workspace.energy_silver."
    "open_meteo_forecast_asof_hourly"
)

asof_row_count = (
    forecast_asof_df.count()
)

asof_decision_count = (
    forecast_asof_df
    .select(
        "system_id",
        "decision_time_utc"
    )
    .distinct()
    .count()
)

leakage_count = (
    forecast_asof_df
    .filter(
        F.col("available_at_utc")
        > F.col("decision_time_utc")
    )
    .count()
)

invalid_horizon_count = (
    forecast_asof_df
    .filter(
        (
            F.col(
                "forecast_horizon_minutes"
            ) <= 0
        )
        |
        (
            F.col(
                "forecast_horizon_minutes"
            ) > 1440
        )
    )
    .count()
)

duplicate_count = (
    forecast_asof_df
    .groupBy(
        "system_id",
        "decision_time_utc",
        "forecast_valid_time_utc"
    )
    .agg(
        F.count("*").alias("row_count")
    )
    .filter(F.col("row_count") > 1)
    .count()
)

incorrect_curve_size_count = (
    forecast_asof_df
    .groupBy(
        "system_id",
        "decision_time_utc"
    )
    .agg(
        F.count("*").alias(
            "forecast_point_count"
        )
    )
    .filter(
        F.col("forecast_point_count")
        != EXPECTED_FORECAST_POINTS_PER_DECISION
    )
    .count()
)

forecast_available_before_run_count = (
    forecast_asof_df
    .filter(
        F.col("available_at_utc")
        < F.col("run_time_utc")
    )
    .count()
)

test_results = {
    "asof_row_count":
        asof_row_count,
    "asof_decision_count":
        asof_decision_count,
    "leakage_count":
        leakage_count,
    "invalid_horizon_count":
        invalid_horizon_count,
    "duplicate_count":
        duplicate_count,
    "incorrect_curve_size_count":
        incorrect_curve_size_count,
    "forecast_available_before_run_count":
        forecast_available_before_run_count,
}

for test_name, test_value in (
    test_results.items()
):
    print(
        f"{test_name}: {test_value}"
    )

test_errors = []

if (
    asof_row_count
    != EXPECTED_ASOF_ROW_COUNT
):
    test_errors.append(
        "Niepoprawna liczba rekordów as-of."
    )

if (
    asof_decision_count
    != EXPECTED_DECISION_COUNT
):
    test_errors.append(
        "Niepoprawna liczba decyzji."
    )

zero_expected_tests = {
    "leakage_count":
        leakage_count,
    "invalid_horizon_count":
        invalid_horizon_count,
    "duplicate_count":
        duplicate_count,
    "incorrect_curve_size_count":
        incorrect_curve_size_count,
    "forecast_available_before_run_count":
        forecast_available_before_run_count,
}

for test_name, test_value in (
    zero_expected_tests.items()
):
    if test_value != 0:
        test_errors.append(
            f"{test_name} = {test_value}"
        )

if test_errors:
    raise RuntimeError(
        "Testy Silver pogody niezaliczone: "
        + "; ".join(test_errors)
    )

print(
    "Wszystkie miesięczne testy "
    "point-in-time zostały zaliczone."
)